# Example 01: the full RT-GSFit pipeline for ST40

This notebook runs every step of RT-GSFit v2 for ST40 pulse 14926. It builds the preshot with GSFit,
stores it in MDSplus, compiles it into the C library, replays the pulse and stores the results. Each step
is one call to the Python package `rtgsfit`, the same call that the `rtgsfit` command line makes,
followed by a plot or a check of what it produced.

RT-GSFit is split in two ([manual, section 1](../../documentation/manual.md#1-overview)):

* the **preshot**, computed in Python before the pulse: everything that does not depend on the
  measurements, such as the Greens functions, the LU factors of the Grad-Shafranov operator and the
  matrices of the least-squares fit;
* **`rtgsfit()`**, a C function compiled together with the preshot. The PCS calls it once per cycle with
  the latest measurements and PF coil currents, and each call returns the equilibrium.

A **replay** calls the compiled `rtgsfit()` from Python on the measurements of a past pulse, exactly as
the PCS would.

```text
Preshot: GSFit ──► preshot ──► RTGSFIT 99000230 TEST03 ──► C source ──► librtgsfit.so
Replay:  ST40 14926 ──► meas, coil_signal ──► rtgsfit() at each time ──► RTGSFIT 52014926 TEST03
```

| Section | Step | Functions | Writes to MDSplus |
|---|---|---|---|
| 1 | Choose the pulses and runs; check the environment | `helpers.check_environment`, `helpers.check_run_is_free` | |
| 2 | Settings | `load_settings`, `Settings.copy_with` | |
| 3 | Build the preshot with GSFit; plot its Greens functions | `build_preshot`, `helpers.gsfit_geometry` | |
| 4 | Store the preshot | `write_preshot` | `RTGSFIT` 99000230 `TEST03` |
| 5 | Read the preshot back and compile it | `read_preshot`, `build_library` | |
| 6 | Read the measurements of the pulse | `read_signals` | |
| 7 | Replay the pulse | `replay` | |
| 8 | Plot the results | `helpers.plot_*` | |
| 9 | Store the results | `write_results` | `RTGSFIT` 52014926 `TEST03` |
| 10 | Read the results back | `read_mdsplus_nodes` | |
| 11 | Next steps: the command line and the PCS build | | |

> **This notebook writes to the shared MDSplus server.** It needs the Tokamak Energy network and write
> access to the `RTGSFIT` tree (see [README.md](README.md)). Section 1 stops before anything is written
> if either run already exists.

The checks and plots are in [helpers.py](helpers.py), so that this notebook shows only the pipeline.
Clear the outputs before committing a change to this notebook: they contain ST40 data.

In [ ]:
import os

# One OpenBLAS thread, as on a real-time core (manual, section 6.5). This must be set before NumPy or
# librtgsfit.so load OpenBLAS, which is why it is the first cell.
print(f"OPENBLAS_NUM_THREADS = {os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')}")

In [ ]:
import helpers
import mdsthin
import numpy as np
from rtgsfit import data_dictionary as dd
from rtgsfit import load_settings
from rtgsfit.database_readers.rtgsfit_mdsplus import read_mdsplus_nodes
from rtgsfit.library import BuildOptions
from rtgsfit.library import build_library
from rtgsfit.library import c_directory
from rtgsfit.replay import build_preshot
from rtgsfit.replay import read_preshot
from rtgsfit.replay import read_signals
from rtgsfit.replay import replay
from rtgsfit.replay import write_preshot
from rtgsfit.replay import write_results
from rtgsfit.results import Results

## 1. Pulses and runs

RT-GSFit stores everything in the `RTGSFIT` MDSplus tree, one *run* per write, at
`\RTGSFIT::TOP.<run name>` ([manual, section 9](../../documentation/manual.md#9-storage-mdsplus-and-local-files)). This
example writes two runs, both called `TEST03`:

* the preshot, at `RTGSFIT` pulse 99000230;
* the replay of ST40 pulse 14926, at `RTGSFIT` pulse 52014926: a replay is stored at 52 000 000 plus the
  ST40 pulse number.

Each run has a description, stored at `\RTGSFIT::TOP.<run name>:HELP`.

In [ ]:
ST40_PULSE = 14_926  # the pulse to replay; GSFit also uses its machine description for the preshot
PRESHOT_PULSE = 99_000_230  # RTGSFIT pulse for the preshot
RESULTS_PULSE = 52_000_000 + ST40_PULSE  # RTGSFIT pulse for the replay
RUN_NAME = "TEST03"  # the run at both pulses
OVERWRITE = False  # True: write over existing runs called RUN_NAME

PRESHOT_DESCRIPTION = f"Testing example_01 for rtgsfit_v2: preshot built by GSFit for ST40 pulse {ST40_PULSE}"
REPLAY_DESCRIPTION = (
    f"Testing example_01 for rtgsfit_v2: replay of ST40 pulse {ST40_PULSE} "
    f"with the preshot from RTGSFIT {PRESHOT_PULSE} {RUN_NAME}"
)

Check the packages, the MDSplus server and the compiler, and that neither run exists yet. To run this
notebook again, choose a new `RUN_NAME`, for example `TEST04`. `OVERWRITE = True` skips the check and
writes over the existing runs.

In [ ]:
helpers.check_environment()
helpers.check_run_is_free(PRESHOT_PULSE, RUN_NAME, OVERWRITE)
helpers.check_run_is_free(RESULTS_PULSE, RUN_NAME, OVERWRITE)

## 2. Settings

Every choice RT-GSFit makes is a setting. The settings are JSON files in
[python/rtgsfit/settings](../../python/rtgsfit/settings), checked against one schema
([manual, section 8](../../documentation/manual.md#8-settings)). `load_settings("st40")` loads
[st40/rtgsfit.json](../../python/rtgsfit/settings/st40/rtgsfit.json). `copy_with` returns a copy with
some settings replaced, each given by its slash-separated path. Every copy is validated, so a misspelt
path or a wrong type fails here rather than halfway through the pipeline.

The two halves of the pipeline use different settings:

* `preshot_settings` builds the preshot with the `preshot/builder` (GSFit) and writes it to `PRESHOT_PULSE`;
* `replay_settings` reads the preshot back from `PRESHOT_PULSE` with the `preshot/reader`, replays
  `replay/pulse` and writes the results to `RESULTS_PULSE`.

Nothing is written to MDSplus by accident: the `rtgsfit_mdsplus` writer must be listed in
`writers/names` *and* `writers/rtgsfit_mdsplus/enabled` must be true. `link_best` stays false, so the
`BEST` run is not pointed at these test runs. `legacy_schema` is false because this version of RT-GSFit
writes the preshot. The default ST40 preshot, `RTGSFIT` 52014922 `RUN21`, was written by an earlier v2
and needs `true` ([manual, section 11.5](../../documentation/manual.md#115-original-v2-preshots-and-fit-names)).

In [ ]:
st40 = load_settings("st40").copy_with({"replay/pulse": ST40_PULSE})

preshot_overrides = {
    "writers/names": ["rtgsfit_mdsplus"],
    "writers/rtgsfit_mdsplus/enabled": True,
    "writers/rtgsfit_mdsplus/link_best": False,
    "writers/rtgsfit_mdsplus/pulse": PRESHOT_PULSE,
    "writers/rtgsfit_mdsplus/run_name": RUN_NAME,
    "writers/rtgsfit_mdsplus/run_description": PRESHOT_DESCRIPTION,
}
replay_overrides = {
    "preshot/reader": "rtgsfit_mdsplus",
    "preshot/rtgsfit_mdsplus/pulse": PRESHOT_PULSE,
    "preshot/rtgsfit_mdsplus/run_name": RUN_NAME,
    "preshot/rtgsfit_mdsplus/legacy_schema": False,
    "writers/names": ["rtgsfit_mdsplus"],
    "writers/rtgsfit_mdsplus/enabled": True,
    "writers/rtgsfit_mdsplus/link_best": False,
    "writers/rtgsfit_mdsplus/pulse": RESULTS_PULSE,
    "writers/rtgsfit_mdsplus/run_name": RUN_NAME,
    "writers/rtgsfit_mdsplus/run_description": REPLAY_DESCRIPTION,
}
preshot_settings = st40.copy_with(preshot_overrides)
replay_settings = st40.copy_with(replay_overrides)

# Settings that both halves share, from st40/rtgsfit.json
for path in ("preshot/builder", "gsfit/settings_path", "signals/reader", "build/timing", "replay"):
    print(f"{path:<20}{st40.get_path(path)}")

## 3. Build the preshot with GSFit

`build_preshot` runs the `preshot/builder`: GSFit, with its `st40_setup_for_rtgsfit` settings
(`gsfit/settings_path`) and the ST40 machine description for `replay/pulse`. GSFit provides the physics:
the grid, the limiter, the Greens functions of the PF coils, passive structures and sensors, and the
uncertainty of every measurement. `assemble_preshot` then derives everything else that the C code needs,
for example the LU factors of the Grad-Shafranov operator and the weighted matrices of the fit. `Preshot`
checks every node against the data dictionary ([manual, section 5](../../documentation/manual.md#5-the-preshot)). GSFit logs
each step as it goes. This is the slowest step of the notebook, taking under a minute.

In [ ]:
preshot = build_preshot(preshot_settings)
print(preshot["INFO:SOURCE"])
print(preshot.summary())
print(f"Content hash: {preshot.content_hash()}")
print(f"Fit unknowns: {list(preshot['FIT:DOF_NAMES'])}")

The sizes are the `SIZES` group of the preshot; in C they are `enum` constants. At every call,
`rtgsfit()` fits `N_FIT_DOF` unknowns to `N_FIT_TARGET` equations. The unknowns are `N_PLASMA` amplitudes
of the plasma current and `N_PASSIVE` passive currents: passive structures and eigenmodes of the
in-vessel components (`IVC_EIG_*`). The equations are the `N_CONSTRAINT` magnetic constraints and
`N_REG` regularisation rows, which penalise passive currents that the measurements do not determine
([manual, section 4.2](../../documentation/manual.md#42-current-model)).

The content hash covers every node that is compiled into C. Section 5 uses it to check the copy read back
from MDSplus.

In [ ]:
figure = helpers.plot_preshot(preshot)

**Left:** the grid. Blue points mark the fixed region where plasma current is geometrically allowed:
the ELMAG vessel interior plus a narrow allowance around limiter samples. Grey points are excluded,
not just the grid edge. Orange edge points carry the Poisson free-boundary condition. The line is the vessel contour; the two
dots are separate limiter points on the MC coils. **Right:** `INITIAL:PSI_NORM` and `INITIAL:MASK`, the
plasma that the first call of `rtgsfit()` starts from; every later call starts from the result of the
call before.

### The Greens functions of the PF coils and passive structures

At every call, `rtgsfit()` adds the flux of the PF coils and of the fitted passive currents to the plasma
flux ([manual, section 4.6](../../documentation/manual.md#46-total-flux)):

```text
psi = GRID:G_COIL coil_current + psi_plasma + GRID:G_PASSIVE fit_dof[N_PLASMA:]
```

Each column of `GRID:G_COIL` and `GRID:G_PASSIVE` is the flux on the grid of one coil circuit or one
passive degree of freedom (DOF), for a current or DOF of 1. The preshot holds only these Greens functions,
not the conductors that make them, so `gsfit_geometry` collects the filaments of the PF coils and passive
structures from the GSFit controller that built the preshot. `build_preshot` cached the controller, so
this is quick. The geometry is used only by the plots below.

In [ ]:
geometry = helpers.gsfit_geometry(preshot_settings)
passive_dofs = list(preshot["FIT:DOF_NAMES"][preshot["SIZES:N_PLASMA"] :])
ivc_modes = [name for name in passive_dofs if name.startswith("IVC_EIG_")]
n_filaments = {name: len(filaments.r) for name, filaments in geometry.passives.items()}
print(f"PF coil circuits: {geometry.coil_members}")
print(f"Passive structures and their numbers of filaments: {n_filaments}")

In [ ]:
figure = helpers.plot_coil_greens(preshot, geometry)

`GRID:G_COIL`, the flux per ampere of each of the eight PF coil circuits (`COIL:NAMES`). The coils of the
circuit are red and the other PF coils grey. Several circuits are two GSFit coils in series, such as
DIV = DIVT + DIVB. The second line of each title is the circuit's row of `COIL:MATRIX`: the current of the
circuit from the PCS signals in `COIL:SIGNALS`, `coil_current = COIL:MATRIX coil_signal`. MCT carries the
current of the MC wire Rogowski coil, and MCB carries that current plus the MCVC power supply. The PSH
circuit has no signal, so its row is zero: its current is 0 and its flux adds nothing. `CONSTRAINT:G_COIL`
is the same for the sensors: `rtgsfit()` subtracts the PF coil contribution from each measurement before
the fit (step 1 in section 7).

In [ ]:
figure = helpers.plot_passive_greens(preshot, geometry, ivc_modes)

**The eigenmodes of the in-vessel components (IVC).** The IVC is 388 filaments, too many to fit one
current each. GSFit instead describes their currents by the slowest eigenmodes of their L/R circuit
equation: an eigenvector `x` of `M x = tau R x`, where `M` is the mutual inductance matrix of the
filaments and `R` their (diagonal) resistance matrix, is a pattern of filament currents that decays as
`exp(-t / tau)` without changing shape. Induced vessel currents are dominated by the slowest modes, so
the fit keeps the 15 with the largest `tau` (`passive_dof_regularisation.json` in the GSFit settings). The
fit unknown `IVC_EIG_nn` is the amplitude of mode `nn`: each filament carries `IVC_EIG_nn × x[filament]`
amperes, with `x` scaled to a 2-norm of 100.

Each panel shows the pattern of filament currents (squares) and the flux it makes on the grid (colour),
both divided by their largest magnitude, so red and blue are opposite signs. The modes alternate between
up-down symmetric (odd `nn`) and antisymmetric (even `nn`): in mode 01 the whole IVC carries current in
one direction, while in mode 02 the current in the top half is opposite to that in the bottom half, the
kind of current that a vertical movement of the plasma induces. The higher modes have more sign changes
and decay faster.

In [ ]:
figure = helpers.plot_eigenmodes(preshot, geometry)

**Left:** the L/R time constant of each mode, from 11.9 ms for mode 01 down to 1 ms. **Middle:** the net
toroidal current of the mode per ampere of amplitude, the sum of `x`. Only the symmetric modes carry a
large net current; the antisymmetric ones carry about 1 % of their total, because the IVC is not exactly
up-down symmetric. **Right:** the regularisation weight of each mode. The
measurements cannot determine all the passive currents, so the fit has `N_REG` regularisation rows
(`FIT:TARGET_NAMES` `IVC_REG_nn`, ..., with a target of 0), which add `(weight × amplitude)²` to `chi_sq`
([manual, section 4.2](../../documentation/manual.md#42-current-model)). Each row is like a measurement that the amplitude is
`0 ± 1 / weight`. The antisymmetric modes are held to about ±1 A, the symmetric ones to ±21 A (modes 01
to 09) or ±6 A (11 to 15). The weights are the regularisation rows of `FIT:G_PASSIVE`, which is compiled into C.

In [ ]:
figure = helpers.plot_passive_greens(preshot, geometry, [name for name in passive_dofs if name not in ivc_modes], 4)

The other passive structures have one DOF each, a constant current density in A/m², so each filament
carries a current in proportion to its area. They are the cases of the BVL coils (hollow boxes around
BVLT and BVLB), the divertor and high-field-side passive stabilisation rings (`DIVPSR*`, `HFSPSR*`) and
the outer vessel (`OVC`). The currents of the four rings are also constraints of the fit
(`I_ROG_DIVPSRT`, ..., computed from flux loop voltages in section 6). Only the OVC is regularised
(`OVC_REG_00`).

## 4. Store the preshot in MDSplus

`write_preshot` passes the preshot to every writer in `writers/names`, here only `rtgsfit_mdsplus`. The
writer creates the run from `RTGSFIT.csv` in `standard_utility`, which must be identical to the output
of `rtgsfit write-csv` (checked in section 1, and again by the writer). It then writes every node under
`\RTGSFIT::TOP.TEST03.PRESHOT`.

In [ ]:
write_preshot(preshot, preshot_settings)

## 5. Read the preshot back and compile it into C

From here on, the pipeline uses the preshot stored in MDSplus, as the PCS build does. `read_preshot`
reads it with the `preshot/reader` of `replay_settings`. Its content hash must equal that of the preshot
that was written.

In [ ]:
preshot_from_mdsplus = read_preshot(replay_settings)
assert preshot_from_mdsplus.content_hash() == preshot.content_hash(), "the preshot changed in MDSplus"
print(f"RTGSFIT {PRESHOT_PULSE} {RUN_NAME}: content hash {preshot_from_mdsplus.content_hash()}, as written")

`build_library` makes this preshot the active one and compiles it. It writes the preshot to the file
`c/build/preshot.bin` and runs `make -C c PRESHOT_FILE=c/build/preshot.bin`. Make runs the C generator
`c/tools/write_preshot.c`, which writes `c/src/preshot_sizes.h` and `c/src/preshot_data.c`, and then
compiles `c/build/librtgsfit.so`. All three are gitignored. The generator rewrites a file only if its
content changes, so building the same preshot again compiles nothing, and the replay below reuses this
build. There is one active preshot: building another one replaces these files. The ST40 settings set
`build/timing`, so the library records the time taken by each step of `rtgsfit()`.

The paths below are those of the files that the replay uses.

In [ ]:
library = build_library(preshot_from_mdsplus, BuildOptions.from_settings(replay_settings))
c_dir = c_directory() / "src"
print(f"Compiled library: {library}")
print(f"Generated preshot: {c_dir}")

The generator turns the preshot into two C files ([manual, section 6.6](../../documentation/manual.md#66-building)):

* `preshot_sizes.h` holds the sizes as `enum` constants, so every array in `c/src` has a fixed size and
  no memory is allocated at run time;
* `preshot_data.c` holds the nodes that `rtgsfit()` uses as `const` arrays: `PRESHOT.<GROUP>:<NODE>` is
  the C constant `<GROUP>_<NODE>`. It also holds `PRESHOT_CHECKSUM`, a CRC-32 of the data, which
  `Rtgsfit` checks so that a library built from another preshot is never used by mistake.

These two files are all that the PCS build needs from the preshot. The PCS generates them with the same
generator, reading the preshot from MDSplus without Python (section 11). The following cell reads the
files used by the build above.

In [ ]:
sizes_h = c_dir / "preshot_sizes.h"
data_c = c_dir / "preshot_data.c"
print(f"Size header: {sizes_h}")
print(sizes_h.read_text())
data_lines = data_c.read_text().splitlines()
print(f"Data source: {data_c} ({data_c.stat().st_size / 1e6:.1f} MB, {len(data_lines)} lines)")
print("\n".join(data_lines[:6]))
declarations = [line for line in data_lines if line.startswith("const ") and "CHECKSUM" not in line]
print("\n".join(line.split(" =")[0] + ";" for line in declarations))

### How the preshot reaches `rtgsfit()`

The preshot is compiled into the C library once. During replay, Python passes only the measurements,
coil currents and the evolving plasma state to each `rtgsfit()` call:

![Diagram of preshot preparation and replay data flow](preshot_to_rtgsfit.svg)

## 6. Read the measurements of the pulse

`read_signals` uses the `signals/reader`, `st40_mdsplus`, to read the inputs of `rtgsfit()` at every
replay time ([manual, section 7.3](../../documentation/manual.md#73-database-readers)):

* `meas`: the measurements named by `PRESHOT.MEAS:NAMES` (flux loops, Bp probes, Rogowski coils and
  flux loop voltages), from the ST40 `MAG` tree;
* `coil_signal`: the PF power supply currents named by `PRESHOT.COIL:SIGNALS`, from the `PSU` tree.

These are the names the PCS uses, so the replay sees the same signals as the PCS. Every signal is
interpolated linearly onto the replay times.

In [ ]:
signals = read_signals(replay_settings, preshot_from_mdsplus)
print(f"{len(signals.time)} times, t = {signals.time[0] * 1e3:.1f} to {signals.time[-1] * 1e3:.1f} ms")
print(f"meas:        {signals.meas.shape}")
print(f"coil_signal: {signals.coil_signal.shape}, {list(preshot_from_mdsplus['COIL:SIGNALS'])}")

In [ ]:
figure = helpers.plot_signals(signals, preshot_from_mdsplus)

**Top row and bottom left:** `meas`, by type. **Bottom middle:** the nine Rogowski constraints after
sensor replacement, `CONSTRAINT:REPLACEMENT meas` ([manual, section 4.1](../../documentation/manual.md#41-constraints-and-fit_target)).
Three are measured by Rogowski coils. The other six are currents in passive structures, computed as
`I = V / R` from the flux loop voltages. **Bottom right:** the currents in the PF coil circuits,
`COIL:MATRIX coil_signal`; for example, the DIV circuit is the DIVT and DIVB coils in series.

## 7. Replay the pulse

`replay` compiles the library for the preshot (here it finds it in the cache) and calls `rtgsfit()`
once per replay time. It passes `psi_norm` and `mask` from one call to the next, exactly as the PCS does.
Each call is one Picard iteration: the current is fitted on the flux surfaces of the previous call. At
the PCS cycle rate, one iteration per call is enough to follow the equilibrium
([manual, section 4](../../documentation/manual.md#4-the-algorithm)). Each call does these 12 steps, and each step is timed:

| Step | Timing section | What it does | Manual |
|---|---|---|---|
| 1 | `CONSTRAINTS` | Sensor replacement; subtract the PF coil contribution; divide by the uncertainty: `fit_target` | 4.1 |
| 2 | `BASIS` | Basis functions of `j_phi` from the `psi_norm` and `mask` of the previous call | 4.2 |
| 3 | `FIT_MATRIX` | Response of `fit_target` to each basis function and passive current | 4.3 |
| 4 | `LEAST_SQUARES` | Least-squares fit of `fit_dof` with LAPACK `dgelss` | 4.3 |
| 5 | `CURRENT` | `j_phi`, plasma current and current centroid | 4.4 |
| 6 | `RESIDUAL` | Reconstructed constraints and `chi_sq` | 4.4 |
| 7 | `POISSON` | Plasma flux from `j_phi`: free-boundary Poisson solve (von Hagenow and Lackner) | 4.5 |
| 8 | `EXTERNAL_PSI` | Add the flux of the PF coils and passive currents | 4.6 |
| 9 | `NULLS` | Magnetic axis and x-points | 4.7 |
| 10 | `BOUNDARY` | Boundary flux: limited or diverted | 4.8 |
| 11 | `FLOOD_FILL` | `mask` of the plasma core | 4.9 |
| 12 | `NORMALISE` | `psi_norm` for the next call | 4.9 |

`signals` is passed in, so the measurements are not read again. The preshot is not passed in, so
`replay` reads it with the `preshot/reader` and records where it came from in `INPUT.PRESHOT_SRC`.
`verbose=True` would print every call.

In [ ]:
results = replay(replay_settings, signals=signals, verbose=False)
helpers.print_replay_summary(results)

`GLOBAL:STATUS` is 0 when a call succeeds; otherwise it combines bit flags
([manual, section 4.10](../../documentation/manual.md#410-status)). Here the calls at the end of the pulse, once the plasma
current has ramped down, return `BOUNDARY_ABOVE_AXIS`: the boundary flux is above the flux at the
magnetic axis, so there is no plasma core. After a failed call, `psi_norm` and `mask` are left unchanged,
so the next call restarts from the last good equilibrium. The settings `replay/stop_below_ip` and
`replay/stop_on_status` can end a replay early instead.

## 8. Plot the results

`results` holds every stored quantity under its data dictionary name, as in MDSplus and in local `.npz`
files. For example, `results["GLOBAL.IP"]` has one value per call, and `results["PROFILES_2D.R_Z.PSI"]` is
`[N_TIME, N_Z, N_R]`. Every name is listed in [data_dictionary.md](../../documentation/data_dictionary.md).

In [ ]:
n_preshot = sum(key.startswith("PRESHOT.") for key in results.flat)
print(f"{len(results.flat)} nodes, of which {n_preshot} are the copy of the preshot")
print(f"GLOBAL.IP: {results['GLOBAL.IP'].shape}, PROFILES_2D.R_Z.PSI: {results['PROFILES_2D.R_Z.PSI'].shape}")

In [ ]:
figure = helpers.plot_time_traces(results)

The main 0D quantities. Calls with a non-zero status are shaded and not plotted. `rtgsfit()` does not
compute the boundary shape (geometric axis, elongation, triangularity and minor radius): Python computes
it from the boundary after the replay ([manual, section 4.11](../../documentation/manual.md#411-post-processing-python-only)).

In [ ]:
figure = helpers.plot_passive_currents(results, geometry)

The fitted passive DOFs, `FIT.DOF_VALUES[:, N_PLASMA:]`. **Top:** the amplitudes of the IVC eigenmodes.
The symmetric modes do most of the work; the regularisation holds the antisymmetric (dashed) ones near 0.
**Bottom:** the net toroidal current in each passive structure: each fitted DOF times the sum of its
filament currents for a DOF of 1, from `geometry`. For an eigenmode that sum is the middle panel of the
eigenmode plot in section 3; for a constant current density, whose DOF is in A/m², it is the area of the
structure.

In [ ]:
figure = helpers.plot_equilibria(results, times=(0.03, 0.08, 0.13, 0.2))

The equilibrium at four times: toroidal current density `j_phi` (colour), poloidal flux contours (grey),
the boundary (green), the magnetic axis (+) and x-points (×). `j_phi` is non-zero only in the plasma core.

In [ ]:
figure = helpers.plot_constraints(results, time=0.1)

Measured (○) and reconstructed (×) values of every magnetic constraint at 100 ms. The measured values are
after sensor replacement (section 6). `chi_sq` sums the squares of (measured − reconstructed) /
uncertainty.

In [ ]:
figure = helpers.plot_timing(results)

The CPU time of each step of `rtgsfit()` over the successful calls (the boxes show the quartiles), and the
total time of each call. The Poisson solve takes the most time. Compare with the typical values in
[manual, section 6.4](../../documentation/manual.md#64-timing).

## 9. Store the results in MDSplus

`write_results` passes the results to every writer in the `writers/names` of `replay_settings`. The run
holds everything needed to reproduce the replay: the results, a copy of the preshot (`PRESHOT`), and the
effective settings, the code version and the raw `meas` and `coil_signal` (`INPUT`)
([manual, section 7.4](../../documentation/manual.md#74-database-writers)).

In [ ]:
assert np.any(results["GLOBAL.STATUS"] == 0), "no call succeeded: the results are not written"
write_results(results, replay_settings)

## 10. Read the results back

`read_mdsplus_nodes` reads any nodes of a run; `dd.ALL_NODES` is every node in the data dictionary. Every
node read back must equal the results in memory.

In [ ]:
stored = Results(read_mdsplus_nodes(RESULTS_PULSE, RUN_NAME, dd.ALL_NODES))
helpers.compare_results(results, stored)
print(f"INPUT.PRESHOT_SRC: {stored['INPUT.PRESHOT_SRC']}")
for pulse in (PRESHOT_PULSE, RESULTS_PULSE):
    print(f"RTGSFIT {pulse} {RUN_NAME}: {helpers.run_description(pulse, RUN_NAME)}")

Any MDSplus client can read the runs. For example, with `mdsthin`:

In [ ]:
with mdsthin.Connection("smaug") as connection:
    connection.openTree("RTGSFIT", RESULTS_PULSE)
    time = connection.get(f"\\RTGSFIT::TOP.{RUN_NAME}:TIME").data()
    ip = connection.get(f"\\RTGSFIT::TOP.{RUN_NAME}.GLOBAL:IP").data()
peak = np.argmax(np.abs(ip))
print(f"Peak |Ip| = {abs(ip[peak]) * 1e-3:.1f} kA at t = {time[peak] * 1e3:.1f} ms")

## 11. Next steps

**The command line.** The `rtgsfit` command makes the same calls. With a new run name:

```bash
RUN_NAME=TEST04
rtgsfit preshot st40 --set replay/pulse=14926 \
    --set 'writers/names=["rtgsfit_mdsplus"]' --set writers/rtgsfit_mdsplus/enabled=true \
    --set writers/rtgsfit_mdsplus/pulse=99000230 --set writers/rtgsfit_mdsplus/run_name=$RUN_NAME \
    --set 'writers/rtgsfit_mdsplus/run_description=Testing example_01 for rtgsfit_v2: preshot'
rtgsfit replay st40 --quiet --set replay/pulse=14926 \
    --set preshot/rtgsfit_mdsplus/pulse=99000230 --set preshot/rtgsfit_mdsplus/run_name=$RUN_NAME \
    --set preshot/rtgsfit_mdsplus/legacy_schema=false \
    --set 'writers/names=["rtgsfit_mdsplus"]' --set writers/rtgsfit_mdsplus/enabled=true \
    --set writers/rtgsfit_mdsplus/pulse=52014926 --set writers/rtgsfit_mdsplus/run_name=$RUN_NAME \
    --set 'writers/rtgsfit_mdsplus/run_description=Testing example_01 for rtgsfit_v2: replay'
```

Arrays must be given as JSON, in quotes. In Python, `run_replay(replay_settings)` from `rtgsfit.replay`
does sections 5 to 9 in one call, without the plots ([manual, section 7](../../documentation/manual.md#7-the-python-package)).

**The PCS build.** The PCS compiles the same C sources, and generates the preshot files from MDSplus
with the same C generator, without Python ([manual, section 6.6](../../documentation/manual.md#66-building)).
From the repository root:

```bash
make -C c SHOT=99000230 RUN_NAME=TEST03    # c/src/preshot_*.{h,c} from MDSplus, then c/build/librtgsfit.so
```

The generator reads preshots written by this version only, not those of `legacy_schema` runs.

**Further reading.** The [manual](../../documentation/manual.md) describes the algorithm (section 4), the preshot (5), the C
library (6), the settings (8) and the storage (9). [data_dictionary.md](../../documentation/data_dictionary.md) lists every
stored quantity.